# Player detection training — A100

Two stages: **smoke test** (~5 min, verifies plumbing) then **main run** (~2 h).
Upload `dataset_stride2.zip` to Drive root *before* connecting to the runtime.

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive'

In [ ]:
!pip install -q ultralytics

In [ ]:
!mkdir -p /content/players
!unzip -q -o "{DRIVE_DIR}/players_dataset_stride2.zip" -d /content/players
!ls /content/players/dataset_stride2

In [ ]:
import glob

data_yaml = '/content/players/dataset_stride2/data.yaml'
print(open(data_yaml).read())
for split in ['train', 'valid']:
    imgs = glob.glob(f'/content/players/dataset_stride2/{split}/images/*.jpg')
    lbls = glob.glob(f'/content/players/dataset_stride2/{split}/labels/*.txt')
    print(split, len(imgs), 'images,', len(lbls), 'labels')
assert imgs and lbls, 'dataset missing — check unzip cell'

## Stage 1 — Smoke test (~5 min)
If this passes and loss drops, start the main run. Check epoch time in the log:
projected main-run total ≈ epoch_time × 3.5 (bigger model, more epochs).

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8s.pt')
model.train(
    data=data_yaml,
    epochs=3,
    imgsz=640,
    batch=-1,
    fraction=0.05,
    cache='disk',
    workers=8,
    project='/content/runs',
    name='player_smoke',
)

## Stage 2 — Main run (~2 h)
Checkpoints write to local disk; only `best.pt` is copied to Drive at the end.

In [ ]:
model = YOLO('yolov8m.pt')
model.train(
    data=data_yaml,
    epochs=30,
    imgsz=960,
    batch=-1,
    patience=10,
    cache='disk',
    workers=8,
    project='/content/runs',
    name='player_m_960',
)

In [ ]:
import shutil

metrics = model.val()
print('mAP50:', metrics.box.map50)
print('mAP50-95:', metrics.box.map)
shutil.copy('/content/runs/player_m_960/weights/best.pt', f'{DRIVE_DIR}/players_best.pt')
print('saved to', f'{DRIVE_DIR}/players_best.pt')